# Data Cleaning
Clean and filter all three datasets, then save to `data/2_cleaned/`.
1. LGBT_EU:   or  
2. HIV_AIDS: 
3. UNICEF_Immunization: 


| Dataset | Files | Notes |
|---|---|---|
| `lgbt_EU` | 4 CSVs | A study on European Queer adults on various life experiences, like experiencing bigotry. The foundation of this analysis|
| `HIV_AIDS_data` | 6 CSVs (two schemas) | A study on HIV and AIDS prevalence, survivors, deaths, and experiences. Worldwide, will be filtered to EU only |
| `UNICEF_Immunization` | 1 xlsx, many vaccine sheets | A record of rates of immunization on a wide range of various vaccines.  Worldwide, will be filtered to EU only |

**Prerequisite:** `5_download_dataset.ipynb` — all raw files must exist in `data/1_source/`.

**Output:** `data/2_cleaned/` with one CSV per dataset (HIV/AIDS split by schema type).

## Setup

In [ ]:
# !pip install -r ../requirements.txt


In [ ]:
import sys
import re
from pathlib import Path

import pandas as pd

In [ ]:
# Ensure src/ is on the path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_download import *

CLEANED_DIR = PROJECT_ROOT / "data" / "2_cleaned"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Source dir   : {SOURCE_DIR}")
print(f"Cleaned dir  : {CLEANED_DIR}")

---
## 1. LGBT EU Survey

**Goals:**
- Load 4 CSVs (skip `SubsetSize`)
- Drop the `notes` column if present
- Remove rows where `CountryCode == 'Average'`
- Extract the canonical EU country list (used to filter the other datasets)
- Save one cleaned CSV per source file

In [ ]:
LGBT_DIR = SOURCE_DIR / "lgbt_EU"

# Files to process — explicitly exclude SubsetSize
SKIP_LGBT = {"LGBT_Survey_SubsetSize.csv"}

lgbt_files = [
    f for f in sorted(LGBT_DIR.glob("*.csv"))
    if f.name not in SKIP_LGBT
]

print(f"LGBT files to clean: {len(lgbt_files)}")
for f in lgbt_files:
    print(f"  {f.name}")

In [ ]:
def clean_lgbt_csv(path):
    """
    Load and clean one LGBT survey CSV.
    - Drops 'notes' column if present
    - Removes rows where CountryCode == 'Average'
    - Strips leading/trailing whitespace from string columns
    """
    df = pd.read_csv(path, encoding="utf-8", low_memory=False)

    print(f"\n{path.name}")
    print(f"  Raw shape      : {df.shape}")
    print(f"  Columns        : {list(df.columns)}")

    # Drop 'notes' column
    if "notes" in df.columns:
        df = df.drop(columns=["notes"])
        print(f"  Dropped        : 'notes'")

    # Remove 'Average' rows
    before = len(df)
    df = df[df["CountryCode"] != "Average"]
    removed = before - len(df)
    print(f"  Removed 'Average' rows : {removed}")

    # Strip whitespace from all string columns
    str_cols = df.select_dtypes(include="object").columns
    df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

    print(f"  Clean shape    : {df.shape}")
    return df

In [ ]:
lgbt_cleaned = {}

for f in lgbt_files:
    key = f.stem   # e.g. 'LGBT_Survey_ViolenceAndHarassment'
    lgbt_cleaned[key] = clean_lgbt_csv(f)